In [1]:
# Core imports
import os
import shutil
import pandas as pd

from dotenv import load_dotenv

# LangChain components
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
)

In [2]:
# Load API key from .env file
load_dotenv()

if not os.getenv("GEMINI_API_KEY"):
    raise ValueError("Missing GEMINI_API_KEY in .env")

In [3]:
# File path
DATA_PATH = "data_processed/main_harry_potter_cleaned.csv"

# Vector DB settings
CHROMA_DIR = "./chroma_db"
COLLECTION_NAME = "harry_potter"

# Reset database on rerun
RESET_DB = True

In [4]:
# Remove old vector database if reset is enabled
if RESET_DB:
    shutil.rmtree(CHROMA_DIR, ignore_errors=True)
    print("Old Chroma DB removed.")

Old Chroma DB removed.


In [5]:
# Load dataset
df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")

Dataset shape: (5766, 27)


In [6]:
# Convert each row into structured text for embeddings
def create_text(row):

    fields = [
        f"Name: {row.get('name', '')}",
        f"House: {row.get('house', '')}",
        f"Species: {row.get('species', '')}",
        f"Blood Status: {row.get('blood_status', '')}",
        f"Skills: {row.get('skills', '')}",
        f"Loyalty: {row.get('loyalty', '')}",
        f"Source: {row.get('source', '')}"
    ]

    # Remove empty fields
    fields = [f for f in fields if not f.endswith(": ")]

    return "\n".join(fields)


df["text"] = df.apply(create_text, axis=1)

In [7]:
# Convert dataframe rows into LangChain Documents
documents = []

for idx, row in enumerate(df.to_dict(orient="records")):

    documents.append(
        Document(
            page_content=row["text"],
            metadata={
                "id": idx,
                "name": row.get("name"),
                "house": row.get("house"),
                "species": row.get("species"),
                "source": row.get("source")
            }
        )
    )

print(f"Documents created: {len(documents)}")

documents = documents[:20]

Documents created: 5766


In [8]:
# Split documents into smaller chunks for better retrieval
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

split_docs = splitter.split_documents(documents)

print(f"Chunks created: {len(split_docs)}")

Chunks created: 20


In [9]:
# Embedding model (Google Gemini embeddings)
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [10]:
# Create Chroma vector database
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR
)

# Store documents only if DB is empty (prevents duplicates + quota spam)
if vectorstore._collection.count() == 0:
    vectorstore.add_documents(
        documents=split_docs,
        ids=[str(i) for i in range(len(split_docs))]
    )
    print("Documents embedded and stored.")
else:
    print("Vector DB already exists. Skipping embedding.")

Documents embedded and stored.


In [11]:
# Retriever with MMR to reduce redundant results
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5}
)

In [12]:
# Format retrieved context for the LLM
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [13]:
# Chat model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [14]:
# RAG prompt
template = """
You are a Harry Potter assistant.

Use ONLY the provided context.

If you do not find the answer, say you don't know based on the provided data.

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate.from_template(template)

In [15]:
# RAG pipeline: retrieve → format → prompt → LLM
chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [16]:
# Test the system
question = "Which characters are in Gryffindor?"

response = chain.invoke(question)

print("QUESTION:")
print(question)

print("\nANSWER:")
print(response)

QUESTION:
Which characters are in Gryffindor?

ANSWER:
Harry James Potter, Fred Weasley, Remus John Lupin, Lavender Brown


In [17]:
# Inspect retrieved documents
retrieved_docs = retriever.invoke(question)

print("\nRETRIEVED DOCUMENTS")
print("=" * 50)

for i, doc in enumerate(retrieved_docs, 1):

    print(f"\nDOC {i}")
    print("-" * 40)
    print(doc.page_content)

    print("\nMETADATA:")
    print(doc.metadata)


RETRIEVED DOCUMENTS

DOC 1
----------------------------------------
Name: Harry James Potter
House: Gryffindor
Species: Human
Blood Status: Half-blood
Skills: Parseltongue| Defence Against the Dark Arts | Seeker
Loyalty: Albus Dumbledore | Dumbledore's Army | Order of the Phoenix | Hogwarts School of Witchcraft and Wizardry
Source: characters

METADATA:
{'name': 'Harry James Potter', 'species': 'Human', 'id': 7, 'source': 'characters', 'house': 'Gryffindor'}

DOC 2
----------------------------------------
Name: Fred Weasley
House: Gryffindor
Species: Human
Blood Status: Pure-blood
Skills: Beater
Loyalty: Dumbledore's Army | Order of the Phoenix | Hogwarts School of Witchcraft and Wizardry
Source: characters

METADATA:
{'species': 'Human', 'name': 'Fred Weasley', 'source': 'characters', 'id': 2, 'house': 'Gryffindor'}

DOC 3
----------------------------------------
Name: Remus John Lupin
House: Gryffindor
Species: Werewolf
Blood Status: Half-blood
Skills: Exceptionally gifted in Defenc